# Day 28 — Weekly and Phase 1 Review: Checkpoint Decision and Week 5 Plan (21 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: check Week 4 and the whole of Phase 1 against what your repo shows, apply the Day 28 checkpoint, and set up Week 5.

- Deliverable: day28/day28_notes.ipynb with your checklist, predictions and recap answers filled in; every local day folder pushed to GitHub; the customer case mock done in the project chat; your real daily study hours.

- Where this sits: end of Week 4 and of Phase 1 (Foundation). Phase 2 (Core AI Engineering) starts with the Claude API as soon as the foundation is in the repo. Today decides when that is.

Time split (5h): a review day is a full working day. Most of it goes into pushing and finishing work, because the checkpoint is decided by the repo.

| Block | Time | What you do |
|---|---|---|
| Review | 1h | Recap, Week 4 table, Phase 1 exit check, checklist, mixed review |
| Checkpoint and plan | 0.5h | Read the scores, the checkpoint decision and the Week 5 plan; write down your real hours |
| Push and finish | 2h | Push every local folder from Days 4 to 27, finished or not; then finish Day 4 |
| DSA | 0.5h | day04/dsa.py: top words, balanced brackets |
| Career | 1h | Customer case mock in chat (40 min), LinkedIn action (20 min) |

Open items first: the repo on GitHub still has two commits, the last one from 28 Oct. Days 1 to 3 are finished. Day 4 is starter files with 8 functions still raising NotImplementedError. Nothing from Days 5 to 27 is there. The starter folders for those days reached you as patch and zip files, because the scheduled task that writes these sessions cannot push to your repo. If you have been working on your laptop, push all of it today, before anything else.

## Recap (2 minutes, from memory)

- On Day 27, redact() runs inside JsonFormatter, not at each logger.info() call. Why there?

- A caller sends the header X-Request-ID with a newline and a fake JSON log line inside it. What could go wrong if you logged it as it is, and what does choose_request_id() do instead?

- Why does export_openapi() write the file with sort_keys=True and indent, and what does test_openapi_current.py catch?

Answers are on the last page of today's session.

## Week 4 in one table

Six days, six habits. Say the right-hand column out loud without looking. Any row you cannot explain to a colleague in a minute goes on your revision list.

| Day | Big idea | The line to remember |
|---|---|---|
| 22 | HTTP and REST: methods, status codes, headers, idempotency, pagination, curl | Status codes are a contract; retry only what is safe to retry |
| 23 | FastAPI routes, Pydantic models, validation errors, /docs | Type hints become validation, docs and response filtering |
| 24 | Postgres behind the API: settings, pool, dependencies, repository | One request, one pooled connection, one transaction |
| 25 | Testing APIs and one error shape | Test through HTTP on a guarded test database; every error looks the same |
| 26 | Webhooks, HMAC signatures, idempotency, API keys | Verify the raw bytes, dedupe the event, store only key hashes |
| 27 | Phase 1 milestone: uv, JSON logs, OpenAPI, README, demo | If a customer cannot run it from the README, it is not done |

The thread through the week: a customer system talks to yours over HTTP. Week 4 put an API in front of the Week 3 database, tested it through HTTP, accepted signed webhooks from a CMS, and packaged it so someone else can run it. That is the shape of most FDE integration work, and Phase 2 adds a model call in the middle of it.

## Phase 1 in review

Phase 1 had one milestone: a working REST API with a database. Four weeks taught the pieces: Python from zero (Week 1), Python for production with Git, the shell, tests, packaging and async (Week 2), SQL and Postgres (Week 3), and HTTP and FastAPI (Week 4). The table below is the exit check. Each line names the evidence a reviewer would look for and what the repo shows today.

| Exit criterion | What proves it | Repo today |
|---|---|---|
| Python that reads and writes files and JSON safely | day04 main.py runs match its docstring | Starter files, 8 TODOs |
| Tests you can trust | A green pytest run on your own code | No test files |
| SQL fluency | Day 15 to 20 queries return the expected results | No .sql files |
| A REST API with a database | content-sync-api with Postgres and green tests | Not in the repo |
| Git as a daily habit | Small commits most days, a PR or two | 2 commits, the last on 28 Oct |
| Showing your work | A README a stranger can follow and a 3-minute demo | README with Days 1 to 3 |

Read this as a statement about the repo, not about you. I cannot see your laptop. But a hiring manager cannot either, and neither can the next run of this plan, which decides what Day 29 teaches.

## Self-assessment checklist

Tick only what you can do without notes or search. Copy the list into the notebook and mark each line yes or no. A no tells us exactly what to revise.

- Read a curl -v output and name the request line, the status line and three headers.

- Pick the right status code for create, delete, a missing entry, a duplicate slug and a bad field, and explain 401 vs 403.

- Explain safe vs idempotent, and why a POST that creates something takes an Idempotency-Key.

- Explain why cursor pagination stays correct on a table that changes while you page, and offset does not.

- Write a FastAPI route with a path parameter, a validated query parameter (Annotated with Query(ge=..., le=...)) and a body model.

- Split EntryCreate and EntryOut, and say what the response model protects.

- Write a PATCH with model_dump(exclude_unset=True) and tell a field that was not sent apart from one sent as null.

- Load settings from environment variables with pydantic-settings and keep secrets in SecretStr.

- Open a connection pool in the lifespan and use a yield dependency that commits or rolls back.

- Test a route with TestClient, a guarded test database and dependency_overrides.

- Verify a webhook signature over the raw body with hmac.compare_digest() and a timestamp tolerance, and make the handler safe to receive the same event twice.

- Store API keys as hashes, log JSON lines with a request id, and export openapi.json so the same code always gives the same file.

## Check your understanding (predict first, then run)

Six questions from across Week 4, in one file. Write every prediction in the notebook before you run anything. If one surprises you, find the line that caused it and explain it in one sentence. Save it as day28/review.py and run it inside your Day 23 project with uv run python review.py, or with any virtual environment that has fastapi[standard] installed. Question 5 prints two lines.

In [ ]:
# Week 4 mixed review: predict every numbered print, then run the file.
# Run it inside your Day 23 project: uv run python review.py
import hashlib
import hmac
import json
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, Query
from fastapi.testclient import TestClient  # in-process client, no server
from pydantic import BaseModel

app = FastAPI()
ENTRIES = {"home": {"uid": "home", "title": "Home", "secret": "x1"}}


class EntryOut(BaseModel):  # used as the return type (response model)
    uid: str
    title: str


class EntryPatch(BaseModel):  # PATCH body: every field optional
    title: str | None = None
    summary: str | None = None


# 1) route order: FastAPI tries routes in the order they were added
@app.get("/entries/{uid}")
def get_entry(uid: str) -> EntryOut:
    if uid not in ENTRIES:
        raise HTTPException(status_code=404, detail="entry not found")
    return EntryOut(**ENTRIES[uid])  # ENTRIES also holds a "secret" key


@app.get("/entries/search")  # registered after the {uid} route
def search() -> dict:
    return {"results": []}


# 3) query validation with Annotated + Query(ge=..., le=...)
@app.get("/entries")
def list_entries(limit: Annotated[int, Query(ge=1, le=50)] = 10) -> dict:
    return {"limit": limit}


# 4) PATCH: the client sends only "summary", and sends it as null
@app.patch("/entries/{uid}")
def patch_entry(uid: str, body: EntryPatch) -> dict:
    return {"changes": body.model_dump(exclude_unset=True)}


# 5) a dependency that tests can swap out
def get_tenant() -> str:
    return "acme-prod"


@app.get("/whoami")
def whoami(tenant: Annotated[str, Depends(get_tenant)]) -> dict:
    return {"tenant": tenant}


client = TestClient(app)

r = client.get("/entries/search")
print(1, r.status_code, r.json())

r = client.get("/entries/home")
print(2, r.json())

r = client.get("/entries", params={"limit": 0})
print(3, r.status_code, r.json()["detail"][0]["loc"])

r = client.patch("/entries/home", json={"summary": None})
print(4, r.json())

app.dependency_overrides[get_tenant] = lambda: "acme-test"
print(5, client.get("/whoami").json())
app.dependency_overrides.clear()  # FDE TIP: clear after every test
print(5, client.get("/whoami").json())

# 6) HMAC over the raw body vs over the body parsed and dumped again
secret = b"whsec_demo"
raw = b'{"uid": "home", "title": "Home"}'  # bytes exactly as sent
good = hmac.new(secret, raw, hashlib.sha256).hexdigest()
redumped = json.dumps(json.loads(raw), separators=(",", ":")).encode()
again = hmac.new(secret, redumped, hashlib.sha256).hexdigest()
print(6, hmac.compare_digest(good, again))

Write each prediction in the form "1: status ..., body ...". Then run the file and compare line by line.

## Progress scores

Scores are 1 to 10 against the bar for a hireable FDE, where 7 means you could do it in an interview without help. They come from what the repo and your session answers show, not from what was taught. Four weeks of topics have been taught and the repo shows the first three days, so the scores have not moved since Day 7.

| Area | Day 21 to Day 28 | Evidence (repo and sessions) | Next step |
|---|---|---|---|
| Python | 3 to 3 | Days 1 to 3 pushed; Day 4 starters untouched | Day 4 on Day 29; Days 30 to 33 |
| TypeScript / Node | 6 to 6 | 7 years of React; not tested yet | Node services in Phase 3 |
| DSA | 2 to 2 | 4 easy problems solved; about 35 open | Day 4's two today; 30 minutes daily |
| SQL and databases | 1 to 1 | Six days of SQL taught; no .sql file in the repo | Day 31 |
| Backend & APIs | 1 to 1 | HTTP and FastAPI taught in Week 4; nothing pushed | Days 32 and 33 |
| LLM application engineering | 1 to 1 | Not started | First Claude API call on Day 34; Week 6 |
| RAG | 1 to 1 | Embedding theory from the IIIT-B programme; no hands-on | Weeks 7 and 8, Project 1 |
| Agents & MCP | 1 to 1 | Not started | Weeks 10 and 11 |
| Evals & reliability | 1 to 1 | pytest and API tests taught; no tests in the repo | Tests from Day 30; evals in Week 9 |
| Cloud & deployment | 1 to 1 | Git, shell, uv taught; 2 commits | Docker and cloud in Weeks 11 and 12 |
| Enterprise integration | 5 to 5 | 7 years of CMS integrations; the webhook receiver is not in the repo yet | Day 32; Project 2 |
| Data skills | 2 to 2 | pandas and JSONB taught; not practised in the repo | Day 30 export cleanup |
| Customer-facing skills | 5 to 5 | 14 years of delivery; no mock run yet, so still provisional | Customer case mock today |
| System design for AI | 1 to 1 | Not started | Week 13 |
| Job hunt & positioning | 2 to 2 | Public repo; journey post and headline not confirmed | LinkedIn action below |

If you have done some of this on your laptop, push it and say so in the Day 28 review: the scores go up the same day, and the checkpoint below is checked again on Day 29.

## Checkpoint decision

Day 21 set this rule: if the repo still shows nothing after Day 4 on Day 28, Week 5 becomes a consolidation week. You rebuild one thin project end to end (CMS export, Postgres, FastAPI) instead of starting the Claude API, and job applications move from Week 8 to Week 10.

Decision: the rule applies. The repo shows nothing after Day 4. Week 5 is a consolidation week, and applications start in Week 10, on 27 Dec. This is the better outcome of the two. Applying in Week 8 with an empty repo would spend your first applications, and FDE hiring managers open GitHub before they finish the CV.

One way to reverse it. The run that writes Day 29 checks the repo first. If day27/content-sync-api is on main with its tests, the consolidation week is skipped: the Week 6 table below moves up to Days 29 to 35 and the original timeline stands, with applications in Week 8. Push by 7 AM IST tomorrow for that check to see it.

Adding a week has a cost, so here is where it comes from. Phase 4 (discovery, scoping, demos, proposals) shrinks from three weeks to two, because customer work is the area where you start strongest. The end date stays 13 Feb 2027.

| Phase | Planned weeks | Revised weeks | What changes |
|---|---|---|---|
| 1. Foundation | 1 to 4 | 1 to 5 | Week 5: one thin project, pushed daily |
| 2. Core AI Engineering | 5 to 8 | 6 to 9 | Project 1 (RAG app) by 26 Dec |
| 3. Agents & Production | 9 to 11 | 10 to 12 | Applications start with Week 10 |
| 4. Portfolio & Customer Skills | 12 to 14 | 13 to 14 | Two weeks instead of three |
| 5. Interview Sprint | 15 to 16 | 15 to 16 | No change |

## Honest calibration: the ₹1 Cr target

The calibration from Day 1 holds. A package near ₹1 Cr in India is a senior offer at a top AI company or one of a few well-funded AI startups, and those loops expect shipped LLM systems, strong system design and solid coding rounds. Sixteen weeks from zero Python does not reach that bar. Sixteen weeks can reach a strong first FDE, AI engineer or solutions engineer role, with three portfolio projects and your CMS background as the edge. ₹1 Cr stays a 12 to 24 month goal from there.

The harder fact is pace. The repo shows three days of work in 31 calendar days. At that rate, even the realistic target slips, because Project 1 needs about two weeks of steady building. Week 5 is built to test one thing: whether a smaller daily target, pushed every day, works for you. Six small pushes this week would change the picture more than any new topic.

## Week 5 plan: consolidation, one thin project end to end

One small project, built one layer a day, reusing what Weeks 1 to 4 taught: fetch entries from a mock CMS, store them in Postgres, serve them over FastAPI, then test, package and demo it. Each day is shorter than a Week 4 day and ends with a push. Each day's folder holds that day's layer, and Day 33 assembles them into day33/thin-sync, the way Day 27 brought Day 26's code in. The theory blocks are short refreshers that point back to the day that taught each idea.

Daily split: 1h refresher · 3h build · 0.5h DSA · 0.5h push and notes.

| Day | Date | Topic | Deliverable |
|---|---|---|---|
| 29 | 22 Nov | Restart: push all local work, finish Day 4 (files, JSON, errors) with a first pytest file | Day 4 done and pushed; day29/status.md (one line per day: done, partial, not started) |
| 30 | 23 Nov | Fetch layer: an Entry dataclass, httpx with timeouts and retries, paginated fetch from a mock CMS, tests with MockTransport | day30/fetch.py with 8 or more green tests |
| 31 | 24 Nov | Store layer: one entries table with constraints, a psycopg upsert in one transaction | day31/schema.sql and load.py; a second load changes 0 rows |
| 32 | 25 Nov | Serve layer: FastAPI list with filters and a cursor, get by uid, 404 and 422 shapes, an API key, settings and a pool | day32/api.py with 10 or more TestClient tests |
| 33 | 26 Nov | Ship: one uv project, one command per step, pytest and ruff clean, a README a stranger can follow | day33/thin-sync, all tests green |
| 34 | 27 Nov | Demo and bridge to Phase 2: a 3-minute demo, then a first Claude API call that summarises one entry from your API | Recorded demo; day34/first_call.py |
| 35 | 28 Nov | Weekly review, checkpoint 2 and the Week 6 plan | Review report, coding mock, Week 6 confirmed |

## Week 6 plan (provisional): Claude API and prompting

This is where Phase 2 starts, either on 29 Nov after the consolidation week or tomorrow if the reversal check above passes. The project through the week is a content assistant for the thin-sync API: it summarises, tags and answers questions about CMS entries. Day 35 confirms or adjusts it.

| Day | Date | Topic | Deliverable |
|---|---|---|---|
| 36 | 29 Nov | Claude API foundations: the Messages API, roles, system prompts, tokens and models, the anthropic Python SDK, keys from the environment, errors and retries | day36/ask.py: a CLI that summarises a CMS entry |
| 37 | 30 Nov | Prompt engineering for production: clear instructions, context, examples, XML tags, templates and versions | Prompts for 3 content tasks, compared on 10 cases |
| 38 | 1 Dec | Structured outputs: JSON that matches a schema, Pydantic validation, what to do with a bad answer | day38: an entry tagger that returns validated JSON |
| 39 | 2 Dec | Tool use: tool definitions, the tool-use loop, tool results and errors | day39: an assistant that calls your API as a tool |
| 40 | 3 Dec | Streaming, cost and latency: streamed responses, token counting, prompt caching, choosing a model | day40: a streaming FastAPI endpoint and a cost report |
| 41 | 4 Dec | Mini-project: content assistant API with 15 test cases | day41: assistant endpoint, README, test sheet |
| 42 | 5 Dec | Weekly review and Week 7 plan (embeddings and RAG) | Review report, mock interview |

Both tables are also added to the plan at the top of the doc.

## Mock interview: customer case

Type Mock interview: customer case in the Forward Deployment Engineer project chat. I will play the head of digital at Acme Retail, a fictional retailer that runs its product and marketing content on Contentstack. Her opening line: "After we publish, our site search shows old prices for hours, and the board wants AI answers on the site by next quarter." You have 40 minutes, then a hire or no-hire verdict with specific feedback. This is your strongest area on paper, so the bar is high. This is what gets scored, in order:

- Discovery before solutions: what "old" means in minutes, how many entries and locales, who publishes, what systems sit between the CMS and search, and who owns them.

- Restating the problem in her words with numbers, and getting her to agree before you design anything.

- Splitting it: fix the stale search first (webhook-driven sync, which you built on Day 26), and scope AI answers as a second phase with its own success measure.

- Explaining trade-offs in plain words: webhooks vs a nightly job, what happens when a webhook is missed, what it costs to run.

- Handling one objection she will raise about security or data residency without overpromising.

- Closing with next steps, an owner on each side and a date.

Do the checklist above first. The mock draws on the Day 26 webhook design more than on any code.

## LinkedIn and portfolio action

This week's action is about what a recruiter sees when they click through from your profile.

- Push first. A post that links to an empty repo does more harm than no post.

- If the Day 7 journey post is not live yet, publish it this week. Keep it short: 14 years in frontend and headless CMS, now building integration and AI systems in Python, with a link to the repo.

- Change your headline to name the direction, for example: "Frontend and headless CMS engineer (Contentstack, Drupal) moving into Forward Deployed Engineering | Python, FastAPI, Postgres, LLM apps". Adjust the words, keep the two halves.

- Once a FastAPI project is pushed (Day 33 at the latest), write one post on what you built, with the repo link: for example, how a signed CMS webhook is verified over the raw bytes, and why.

## DSA practice (30 min)

No new problems this week. Finish the two in day04/dsa.py, which is already in your repo with signatures, examples and hints.

- Top k words. top_words(text, k). Hint: build the counts first, then sort the (word, count) pairs once. Ask what key makes Python sort by count high to low and, for equal counts, by word A to Z.

- Balanced brackets. is_balanced(s). Hint: a list works as a stack. Write down by hand what the stack holds after each character of "([)]", and you will see where the check belongs.

## Quiz (3 quick questions)

- Contentstack sends the same publish webhook three times in 30 seconds, and an older event arrives after a newer one. Which two Week 4 ideas keep your database correct, and how?

- One partner gets 401 from your API, another gets 403. What does each status tell them to check?

- Why does the checkpoint look at the repo and not at the hours you studied?

## What you achieved this week

Week 4 covered HTTP and REST from the wire up, FastAPI with Pydantic, Postgres behind an API with settings, a pool and a repository layer, API tests with one error contract, signed webhooks with idempotency and API keys, and the Phase 1 milestone project with logs, docs and a demo. Today you checked Phase 1 against the repo, applied the checkpoint and planned a shorter, push-every-day Week 5.

Tomorrow (Day 29): restart day: push everything you have, finish Day 4 with its first tests, and write status.md, so the rest of Week 5 builds on what is really there.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 28 review": your checklist with yes or no on each line, your recap answers, your six predictions next to the real outputs, the link to your latest commit, your quiz answers, and how many hours a day you can really study next week. Then run the customer case mock in the same chat.

## Recap answers

*Your answers:*
